In [3]:
!pip install -r requirements.txt

In [20]:
!pip install langchain-nvidia-ai-endpoints

In [4]:
with open("./Moby-Dick.txt", "r", encoding='utf-8') as f:
    book= f.read()

In [21]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import getpass


In [11]:
# NVIDIA_API_KEY= "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"

Nhập API_KEY ········


In [22]:
llm = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key="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", 
  temperature=1,
  top_p=1,
)


## Split

In [28]:
text_chunks_chain= (
    RunnableLambda(lambda x:
                   [
                       {
                           'chunk': text_chunks,
                       }
                       for text_chunks in TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
                   ]
    )
)

## Map

In [29]:
summarize_CHUNK_prompt_TEMPLATE = """
    Viết lại vắn tắt những dòng text dưới đây, gồm cả những chi tiết chính
    Text: {chunk}
"""
summarize_CHUNK_prompt = PromptTemplate.from_template(summarize_CHUNK_prompt_TEMPLATE)
summarize_CHUNK_CHAIN = summarize_CHUNK_prompt | llm

summarize_MAP_CHAIN = (
    RunnableParallel( 
        # Nhận 1 chunk → chạy summarize_CHUNK_CHAIN
        # → StrOutputParser() → tạo {"summary": "..."}
        {
            'summary' : summarize_CHUNK_CHAIN | StrOutputParser()
            
            
            
        }
    )

)

## Reduce

In [33]:
sum_sums_from_map_prompt_tem = """
Write a concise summary of the following text, which joins several summaries, and include the main details. 
Text: {summaries}
"""

sum_sums_from_map_prompt = PromptTemplate.from_template(sum_sums_from_map_prompt_tem)
sum_reduce_chain = (
    RunnableLambda(lambda x:
                   {
                       'summaries' : '\n'.join([i['summary'] for i in x]),
                   }
    ) | sum_sums_from_map_prompt | llm | StrOutputParser()
)

## MapReduce

In [34]:
map_reduce_chain = (
    text_chunks_chain | summarize_MAP_CHAIN.map() | sum_reduce_chain
)

In [ ]:
tomtat = map_reduce_chain.invoke(book)

# Tóm tắt các tài liệu (mọi loại định dạng)

In [ ]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader= WikipediaLoader(query= "Messi", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [47]:
print(wikipedia_docs)

[Document(metadata={'title': 'Lionel Messi', 'summary': 'Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains Major League Soccer club Inter Miami. Widely regarded as one of the greatest players in history, he has set numerous records for individual accolades won throughout his professional footballing career, including eight Ballons d\'Or, six European Golden Shoes, and being named the world\'s best player by FIFA eight times. Messi was named the greatest player of the 21st century by ESPN in 2024, and in 2025, he was named the All Time Men\'s World Best Player by the IFFHS. In 2020 and 2023 he was named the Laureus World Sportsman of the Year, becoming the first team-sport athlete to win the award. He was among Time\'s 100 most influential people in the world in 2011, 2012, and 2023.\nMessi is the most decorated player in the history of professional football, having won 46 team trophies across more than 1,150 ca

In [52]:
!pip install python-docx

In [55]:
from docx import Document
doc = Document()

doc.add_heading("Wikipedia - Messi", level=1)

for i, wikipedia_doc in enumerate(wikipedia_docs, start=1):
    doc.add_heading(f"Tài liệu {i}", level=2)

    doc.add_paragraph(
        f"Metadata: {wikipedia_doc.metadata}"
    )

doc.save("tailieu_messi.docx")

print("tao file thanh cong")

tao file thanh cong


In [ ]:
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, TextLoader
word_doc = Docx2txtLoader("tailieu_messi.docx").load()
pdf_file = PyPDFLoader("messi.pdf").load()
all_docs = word_doc+ pdf_doc